In [13]:
import pandas as pd
import numpy as np
#import pymysql
from sqlalchemy import create_engine
import getpass  # To get the password without showing the input
from sqlalchemy import text
password = getpass.getpass()

# Note that when you use _SQLAlchemy_ and establish the connection,
#you do not even need to be logged in Sequel Pro or MySQL Workbench.
# ==========================================
# 1. Establish connection to Sakila database
# ==========================================
db = "sakila"
connection_string = 'mysql+pymysql://root:' + password + '@localhost/'+db
engine = create_engine(connection_string)


connection = engine.connect()
query = text('SELECT * FROM actor')
result = connection.execute(query)

row=result.first()

 ········


In [19]:



# ==========================================
# 2. Function: rentals_month
# ==========================================
def rentals_month(engine, month, year):
    """
    Retrieves rental data for a given month and year from the Sakila database.
    """
    query = """
    SELECT rental_id, rental_date, customer_id, inventory_id, staff_id
    FROM rental
    WHERE MONTH(rental_date) = %s AND YEAR(rental_date) = %s;
    """
    # Execute query and load into a Pandas DataFrame
    df = pd.read_sql(query, con=engine, params=(month, year))
    return df


# ==========================================
# 3. Function: rental_count_month
# ==========================================
def rental_count_month(df_rentals, month, year):
    """
    Counts the number of rentals made by each customer_id during a selected month and year.
    """
    # Format column name with leading zero padding for the month (e.g., 05_2005)
    column_name = f"rentals_{month:02d}_{year}"
    
    # Group by customer_id and count rentals
    df_counts = df_rentals.groupby('customer_id').size().reset_index(name=column_name)
    
    return df_counts


# ==========================================
# 4. Function: compare_rentals
# ==========================================
def compare_rentals(df_month1, df_month2):
    """
    Combines two rental count DataFrames and calculates the difference 
    in activity between the two months for active customers.
    """
    # Extract the custom column names dynamically
    col_m1 = df_month1.columns[1]
    col_m2 = df_month2.columns[1]
    
    # Merge using an inner join to find customers active in BOTH months
    df_combined = pd.merge(df_month1, df_month2, on='customer_id', how='inner')
    
    # Calculate the change in activity (Month 2 - Month 1)
    df_combined['difference'] = df_combined[col_m2] - df_combined[col_m1]
    
    return df_combined


# ==========================================
# Example Execution Workflow
# ==========================================
if __name__ == "__main__":
    # Fetch raw data for May and June 2005
    df_may_raw = rentals_month(engine, 5, 2005)
    df_june_raw = rentals_month(engine, 6, 2005)
    
    # Aggregate rental counts per customer
    df_may_counts = rental_count_month(df_may_raw, 5, 2005)
    df_june_counts = rental_count_month(df_june_raw, 6, 2005)
    
    # Compare activity between both months
    df_final_comparison = compare_rentals(df_may_counts, df_june_counts)
    #df= pd.DataFrame(df_final_comparison.all())
    
    # View the final result
    # print(df_final_comparison.head())
